In [2]:
import datetime
import json
from pathlib import Path

from datasmith.docker.context import ContextRegistry

curr_date: str = datetime.datetime.now().isoformat()

%cd /mnt/sdd1/atharvas/formulacode/datasmith/

21:27:11 WARNING  datasmith: No tokens.env file found. Skipping environment variable setup.
21:27:11 WARNING  simple_useragent.core: Falling back to historic user agent.
21:27:11 WARNING  datasmith: CACHE_LOCATION environment variable not set. Using default 'cache.db'.


/mnt/sdd1/atharvas/formulacode/datasmith


In [3]:
# traverse and find all context_registry*.json files under scratch/**
registries = Path("scratch").rglob("*context_registry*.json")


def merge_with_mtime(merged_json, reg_json, modified_time):
    # when updating, if there is a conflict, prefer the one with the latest modified time
    for key in reg_json["contexts"]:
        if key in merged_json.get("contexts", {}):
            existing_time = merged_json["contexts"][key].get("modified_time", 0)
            if modified_time > existing_time:
                merged_json["contexts"][key] = reg_json["contexts"][key]
                merged_json["contexts"][key]["modified_time"] = modified_time
        else:
            if "contexts" not in merged_json:
                merged_json["contexts"] = {}
            merged_json["contexts"][key] = reg_json["contexts"][key]
            merged_json["contexts"][key]["modified_time"] = modified_time
    # also merge other top-level keys if they don't exist
    for key in reg_json:
        if key != "contexts" and key not in merged_json:
            merged_json[key] = reg_json[key]


merged_json = {}
for reg in registries:
    reg_json = json.loads(reg.read_text())
    modified_time = reg.stat().st_mtime
    print(f"{reg} : {len(reg_json['contexts'])} entries")
    merge_with_mtime(merged_json, reg_json, modified_time)

registry = ContextRegistry.deserialize(payload=json.dumps(merged_json))
len(registry.registry)

scratch/context_registry_init.json : 5 entries
scratch/merged_context_registry_2025-09-04T08:32:08.486247.json : 140 entries
scratch/merged_context_registry_2025-09-05T20:02:28.617179.json : 540 entries
scratch/merged_context_registry_2025-09-06T01:31:46.096023.json : 700 entries
scratch/merged_context_registry_2025-09-04T23:54:53.035665.json : 178 entries
scratch/merged_context_registry_2025-09-06T06:14:21.165351.json : 975 entries
scratch/artifacts/pipeflush/context_registry.json : 7 entries
scratch/artifacts/pipeflush/tiny/context_registry.json : 8 entries
scratch/artifacts/pipeflush/chunk_1/context_registry_venus_bak.json : 367 entries
scratch/artifacts/pipeflush/chunk_1/context_registry_venus.json : 411 entries
scratch/artifacts/pipeflush/chunk_1/context_registry.json : 7 entries
scratch/artifacts/pipeflush/chunk_0/context_registry_shiva.json : 620 entries
scratch/artifacts/pipeflush/chunk_0/context_registry.json : 7 entries
scratch/artifacts/pipeflush/chunk_0/context_registry_shi

1129

In [4]:
registry.save_to_file(Path(f"scratch/merged_context_registry_{curr_date}.json"))

21:27:20 INFO     datasmith.docker.context: Context registry saved to scratch/merged_context_registry_2025-09-06T21:27:11.754109.json
